<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBM3130-PLN-2026/blob/main/semana-08/Aula08_Analise_Sentimento_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 IBM3130 — Processamento de Linguagem Natural
## Aula 08 · Análise de Sentimento

---



---
## ⚙️ Passo 0 — Ambiente

Execute esta célula **uma única vez** no início de cada sessão.

In [ ]:
# Instalações
!pip install spacy scikit-learn -q
!python -m spacy download pt_core_news_sm -q

# Recursos NLTK
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp',      quiet=True)

# Importações principais
import spacy
import matplotlib.pyplot as plt
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.corpus   import stopwords
from nltk.stem     import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes             import MultinomialNB
from sklearn.linear_model            import LogisticRegression
from sklearn.model_selection         import train_test_split
from sklearn.metrics                 import classification_report, confusion_matrix

nlp = spacy.load('pt_core_news_sm')

print('Ambiente pronto!')
print('spaCy:', spacy.__version__)

---
## 🧠 Contextualização

Nas aulas anteriores aprendemos a **representar textos como números**.
Agora vamos usar essas representações para resolver um problema de alto valor econômico:

> *O que as pessoas estão falando sobre meu produto, serviço ou marca?*

Essa é a pergunta central da **Análise de Sentimento**.

### Os três níveis de granularidade

| Nível | Pergunta | Exemplo |
|-------|----------|---------|
| **Documento** | Essa review é positiva ou negativa? | `'Produto ótimo!'` → POSITIVO |
| **Sentença** | Cada frase tem qual sentimento? | `'Câmera boa. Bateria fraca.'` |
| **Aspecto (ABSA)** | Sobre o quê é o sentimento? | câmera=POS · bateria=NEG |

### O que torna a tarefa difícil

```
'Que produto incrível! Quebrou no primeiro dia.'  → Ironia
'Não é ruim.'                                     → Dupla negação
'Chegou antes do prazo!'                          → Sentimento implícito
```

---
# 📌 PARTE 1 — Abordagem Léxica

A abordagem léxica usa um **dicionário de sentimento** — uma lista de palavras
com escores pré-definidos — e soma os escores das palavras encontradas no texto.

**Vantagem:** não precisa de dados rotulados.  
**Desvantagem:** não capta contexto, ironia ou sentimentos implícitos.

### 1.1 — Construindo o léxico de sentimento

In [ ]:
# Léxico de sentimento em português
lexico_positivo = {
    'excelente': 2, 'ótimo': 2, 'maravilhoso': 2, 'incrível': 2,
    'perfeito': 2, 'adorei': 2, 'amei': 2, 'fantástico': 2,
    'bom': 1, 'gostei': 1, 'recomendo': 1, 'rápido': 1,
    'bonito': 1, 'resistente': 1, 'qualidade': 1, 'funciona': 1,
    'satisfeito': 1, 'vale': 1, 'eficiente': 1, 'prático': 1,
}

lexico_negativo = {
    'péssimo': -2, 'horrível': -2, 'terrível': -2, 'lixo': -2,
    'odiei': -2, 'detestei': -2, 'defeituoso': -2, 'desastre': -2,
    'ruim': -1, 'fraco': -1, 'quebrado': -1, 'demorou': -1,
    'arrependido': -1, 'decepcionado': -1, 'problema': -1, 'falha': -1,
    'frágil': -1, 'caro': -1, 'parou': -1, 'decepcionante': -1,
}

# Amplificadores: intensificam a polaridade
amplificadores = {
    'muito': 1.5, 'extremamente': 2.0,
    'super': 1.5, 'bastante': 1.3, 'demais': 1.4,
}

# Negadores: invertem a polaridade
negadores = {'não', 'nunca', 'jamais', 'nem', 'nenhum'}

print(f'Léxico positivo: {len(lexico_positivo)} palavras')
print(f'Léxico negativo: {len(lexico_negativo)} palavras')
print(f'Amplificadores:  {len(amplificadores)}')
print(f'Negadores:       {len(negadores)}')

### 1.2 — Classificador léxico com tratamento de negação

O classificador percorre os tokens e:
- Soma o escore das palavras do léxico
- Aplica o fator dos amplificadores
- **Inverte a polaridade** quando encontra um negador

In [ ]:
def classificar_lexico(texto, lex_pos, lex_neg, amplifs, negs):

    tokens        = word_tokenize(texto.lower(), language='portuguese')
    escore        = 0
    negacao_ativa = False

    # Palavras funcionais que não desativam a negação
    funcional = {'o','a','os','as','um','uma','de','da','do','que','e','é'}

    for i in range(len(tokens)):
        token = tokens[i]

        # É negador?
        if token in negs:
            negacao_ativa = True
            continue

        # Fator de amplificação (palavra anterior)
        fator = 1.0
        if i > 0 and tokens[i-1] in amplifs:
            fator = amplifs[tokens[i-1]]

        # Calcular contribuição
        if token in lex_pos:
            contrib = lex_pos[token] * fator
            escore  = escore + (-contrib if negacao_ativa else contrib)
            negacao_ativa = False

        elif token in lex_neg:
            contrib = abs(lex_neg[token]) * fator
            escore  = escore + (contrib if negacao_ativa else -contrib)
            negacao_ativa = False

        elif token not in funcional:
            negacao_ativa = False

    # Classificar pelo escore final
    if escore > 0:
        return 'POSITIVO', round(escore, 2)
    elif escore < 0:
        return 'NEGATIVO', round(escore, 2)
    else:
        return 'NEUTRO', 0.0

print('Função classificar_lexico definida!')

### 1.3 — Testando o classificador léxico

In [ ]:
reviews_teste = [
    ('O produto é excelente e chegou muito rápido!',          'positivo'),
    ('Péssima qualidade. Quebrou no primeiro dia.',           'negativo'),
    ('Não gostei do produto. Muito fraco para o preço.',      'negativo'),
    ('Recomendo bastante! Funciona perfeitamente.',           'positivo'),
    ('Produto ok. Entrega normal.',                           'neutro'),
]

print('=== Classificação Léxica ===')
print()
print(f'{"Review":<50} {"Esperado":<10} {"Obtido":<10} {"Escore":>8}')
print('-' * 83)

acertos = 0
for texto, esperado in reviews_teste:
    classe, escore = classificar_lexico(
        texto, lexico_positivo, lexico_negativo,
        amplificadores, negadores
    )
    ok      = '✓' if classe.lower() == esperado else '✗'
    acertos = acertos + (1 if classe.lower() == esperado else 0)
    print(f'{ok} {texto[:48]:<49} {esperado:<10} {classe:<10} {escore:>+8.2f}')

print()
print(f'Acertos: {acertos}/{len(reviews_teste)}')

### 1.4 — Casos onde o léxico falha

In [ ]:
casos_dificeis = [
    ('Que produto incrível! Quebrou no primeiro dia.',     'negativo', 'Ironia'),
    ('Não é ruim.',                                        'positivo', 'Dupla negação'),
    ('Chegou antes do prazo!',                             'positivo', 'Sentimento implícito'),
    ('Para o preço, está ótimo.',                          'positivo', 'Sentimento condicional'),
]

print('=== Casos onde o léxico falha ===')
print()

for texto, esperado, problema in casos_dificeis:
    classe, escore = classificar_lexico(
        texto, lexico_positivo, lexico_negativo,
        amplificadores, negadores
    )
    ok = '✓' if classe.lower() == esperado else '✗'
    print(f'{ok} Texto:    "{texto}"')
    print(f'   Esperado: {esperado.upper():<10}  Obtido: {classe}')
    print(f'   Problema: {problema}')
    print()

### 📝 Reflexão — Parte 1

**Pergunta 1:** Para qual dos casos difíceis o léxico errou mais? Por quê?

**Pergunta 2:** Como você modificaria o classificador para tratar a dupla negação `'não é ruim'`?

*(Responda aqui — dê duplo clique para editar)*

---
# 📌 PARTE 2 — Abordagem Supervisionada

Em vez de usar regras e dicionários, deixamos o modelo **aprender** os padrões
de sentimento a partir de exemplos já rotulados.

Pipeline:
```
Corpus rotulado → Pré-processamento → TF-IDF → Classificador → Avaliação
```

### 2.1 — Corpus rotulado

In [ ]:
# Corpus de reviews rotuladas — 1 = positivo, 0 = negativo
reviews = [
    # Positivas
    ('produto excelente chegou rápido funciona perfeitamente', 1),
    ('qualidade incrível vale cada centavo recomendo bastante', 1),
    ('entrega rápida produto bonito resistente adorei', 1),
    ('ótima compra superou expectativas muito satisfeito', 1),
    ('produto perfeito atendimento excelente voltarei comprar', 1),
    ('chegou antes prazo embalagem ótima produto funcionando', 1),
    ('bom custo benefício produto resistente recomendo', 1),
    ('excelente qualidade melhor produto comprei este ano', 1),
    ('produto fantástico superou expectativas entrega rápida', 1),
    ('muito bom produto funciona perfeitamente recomendo', 1),
    ('ótima relação custo benefício produto prático eficiente', 1),
    ('chegou bem embalado produto robusto resistente ótimo', 1),
    ('produto superou todas as expectativas entrega impecável', 1),
    ('adorei cada detalhe produto funciona exatamente prometido', 1),
    ('compra excelente material robusto durável recomendo todos', 1),
    ('envio rápido produto original bem embalado satisfeito', 1),
    # Negativas
    ('produto péssimo parou funcionar dois dias lixo', 0),
    ('horrível chegou quebrado atendimento desastre', 0),
    ('não funciona anunciado qualidade ruim muito frágil', 0),
    ('decepcionado produto defeituoso arrependido compra', 0),
    ('pior compra vida produto péssimo não recomendo', 0),
    ('produto falhou semana problema atendimento terrível', 0),
    ('qualidade horrível quebrou facilmente muito ruim', 0),
    ('não gostei produto fraco preço alto decepcionante', 0),
    ('produto defeituoso devolvi atendimento demorado ruim', 0),
    ('péssima experiência produto parou funcionar jamais compro', 0),
    ('chegou com defeito atendimento péssimo não resolvi', 0),
    ('frágil demais quebrou embalagem entrega horrível', 0),
    ('lixo absoluto quebrou primeiro uso péssima qualidade', 0),
    ('produto falsificado não funciona propaganda enganosa ruim', 0),
    ('arrependido compra material frágil decepcionante caro', 0),
    ('atendimento horrível produto com defeito devolução difícil', 0),
]

textos  = [r[0] for r in reviews]
rotulos = [r[1] for r in reviews]

positivos = sum(rotulos)
negativos = len(rotulos) - positivos

print(f'Total de reviews:   {len(reviews)}')
print(f'  Positivas (1):    {positivos}')
print(f'  Negativas (0):    {negativos}')

### 2.2 — Pré-processamento do corpus

Aplicamos o pipeline completo da Aula 02:
lowercase → tokenizar → filtrar stopwords (preservando `'não'`) → stemming

In [ ]:
stemmer = RSLPStemmer()
stop_pt = stopwords.words('portuguese')
stop_pt.remove('não')   # IMPORTANTE: preservar a negação!

def preprocessar(texto):
    tokens = word_tokenize(texto.lower(), language='portuguese')
    tokens_limpos = []
    for token in tokens:
        if token.isalpha() and token not in stop_pt and len(token) > 2:
            tokens_limpos.append(stemmer.stem(token))
    return ' '.join(tokens_limpos)

# Aplicar em todo o corpus
textos_proc = []
for texto in textos:
    textos_proc.append(preprocessar(texto))

# Mostrar comparação
print('Antes e depois do pré-processamento:')
print()
for i in range(4):
    emoji = '✅' if rotulos[i] == 1 else '❌'
    print(f'{emoji} Original:   "{textos[i]}"')
    print(f'   Processado: "{textos_proc[i]}"')
    print()

### 2.3 — Divisão treino/teste e vetorização TF-IDF

In [ ]:
# Dividir corpus em treino (70%) e teste (30%)
X_train, X_test, y_train, y_test = train_test_split(
    textos_proc, rotulos,
    test_size    = 0.3,
    random_state = 42,
    stratify     = rotulos   # manter proporção de classes
)

print(f'Treino: {len(X_train)} reviews')
print(f'Teste:  {len(X_test)} reviews')
print()

# Vetorizar com TF-IDF — unigramas e bigramas
vetorizador = TfidfVectorizer(
    ngram_range  = (1, 2),   # unigramas + bigramas
    min_df       = 1,        # mínimo de documentos com o termo
    max_features = 300       # limitar vocabulário
)

X_train_vec = vetorizador.fit_transform(X_train)
X_test_vec  = vetorizador.transform(X_test)

print(f'Dimensões da matriz de treino: {X_train_vec.shape}')
print(f'Dimensões da matriz de teste:  {X_test_vec.shape}')
print()

# Mostrar algumas features geradas
features = vetorizador.get_feature_names_out()
print(f'Total de features: {len(features)}')
print(f'Exemplos de unigramas: {[f for f in features if " " not in f][:8]}')
print(f'Exemplos de bigramas:  {[f for f in features if " " in f][:8]}')

### 2.4 — Treinar Naive Bayes e Regressão Logística

In [ ]:
# Classificador 1 — Naive Bayes Multinomial
nb = MultinomialNB(alpha=0.5)  # alpha = suavização de Laplace
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)

# Classificador 2 — Regressão Logística
lr = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

# Exibir resultados comparativos
for nome, y_pred in [('Naive Bayes', y_pred_nb), ('Regressão Logística', y_pred_lr)]:
    print(f'{'=' * 45}')
    print(f'  {nome}')
    print(f'{'=' * 45}')
    print(classification_report(
        y_test, y_pred,
        target_names = ['Negativo', 'Positivo'],
        zero_division = 0
    ))

### 2.5 — Matriz de confusão

In [ ]:
# Calcular matrizes de confusão
mc_nb = confusion_matrix(y_test, y_pred_nb)
mc_lr = confusion_matrix(y_test, y_pred_lr)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, mc, titulo in zip(
    axes,
    [mc_nb, mc_lr],
    ['Naive Bayes', 'Regressão Logística']
):
    im = ax.imshow(mc, cmap='Blues')

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Negativo', 'Positivo'], fontsize=11)
    ax.set_yticklabels(['Negativo', 'Positivo'], fontsize=11)
    ax.set_xlabel('Previsto', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)
    ax.set_title(f'Matriz de Confusão\n{titulo}', fontsize=12, fontweight='bold')

    for i in range(2):
        for j in range(2):
            cor   = 'white' if mc[i, j] > mc.max() / 2 else 'black'
            label = ['TN','FP','FN','TP'][i*2+j]
            ax.text(j, i, f'{mc[i,j]}\n({label})',
                    ha='center', va='center',
                    color=cor, fontsize=13, fontweight='bold')

    plt.colorbar(im, ax=ax)

plt.suptitle('Comparação dos Classificadores', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Legenda
print('TN = Verdadeiro Negativo | FP = Falso Positivo')
print('FN = Falso Negativo      | TP = Verdadeiro Positivo')

### 2.6 — Calcular métricas manualmente

In [ ]:
# Calcular métricas manualmente a partir da matriz de confusão
# Usando Regressão Logística como referência
mc = mc_lr

TN = mc[0, 0]
FP = mc[0, 1]
FN = mc[1, 0]
TP = mc[1, 1]

precisao = TP / (TP + FP) if (TP + FP) > 0 else 0
recall   = TP / (TP + FN) if (TP + FN) > 0 else 0
f1       = 2 * precisao * recall / (precisao + recall) if (precisao + recall) > 0 else 0
acuracia = (TP + TN) / (TP + TN + FP + FN)

print('=== Métricas calculadas manualmente (Regressão Logística) ===')
print()
print(f'  TP (Verdadeiro Positivo): {TP}')
print(f'  TN (Verdadeiro Negativo): {TN}')
print(f'  FP (Falso Positivo):      {FP}')
print(f'  FN (Falso Negativo):      {FN}')
print()
print(f'  Precisão  = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precisao:.4f} ({precisao*100:.1f}%)')
print(f'  Recall    = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall:.4f}   ({recall*100:.1f}%)')
print(f'  F1-Score  = 2×P×R/(P+R)              = {f1:.4f}   ({f1*100:.1f}%)')
print(f'  Acurácia  = (TP+TN)/Total             = {acuracia:.4f} ({acuracia*100:.1f}%)')

### 📝 Reflexão — Parte 2

**Pergunta 1:** Qual dos dois classificadores teve melhor F1-Score? A diferença é significativa?

**Pergunta 2:** Suponha que você precise monitorar uma crise de reputação — reviews negativas que
passam despercebidas custam mais do que classificar uma positiva como negativa.
Qual métrica você priorizaria: Precisão ou Recall? Por quê?

*(Responda aqui)*

---
# 📌 PARTE 3 — ABSA — Análise de Sentimento por Aspecto

A análise de documento diz que a review é negativa.
Mas **o que** é negativo? O preço? A entrega? O produto?

O ABSA identifica o sentimento sobre **cada aspecto específico** usando:
- `token.pos_` → identificar adjetivos (ADJ)
- `token.head` → encontrar o substantivo que o adjetivo modifica
- `token.dep_` → verificar se há negação no contexto

### 3.1 — Inspecionar tokens de uma review com spaCy

In [ ]:
# Inspecionar os atributos de cada token para entender o ABSA
frase = 'A câmera é excelente mas a bateria é muito fraca.'
doc   = nlp(frase)

print(f'Frase: "{frase}"')
print()
print(f'{"Token":<15} {"Lema":<15} {"POS":<8} {"Dep":<10} {"Head"}')
print('-' * 60)

for token in doc:
    if not token.is_space:
        print(f'{token.text:<15} {token.lemma_:<15} {token.pos_:<8} {token.dep_:<10} {token.head.text}')

### 3.2 — Extrator de aspectos com dependências sintáticas

In [ ]:
# Léxico combinado
lexico_sent = {}
for p in lexico_positivo:
    lexico_sent[p] =  1
for p in lexico_negativo:
    lexico_sent[p] = -1

def extrair_aspectos(texto, lexico, negs):

    doc      = nlp(texto.lower())
    aspectos = []

    for token in doc:

        # Localizar adjetivos
        if token.pos_ == 'ADJ':
            adjetivo = token.lemma_.lower()

            # O substantivo que o adjetivo modifica (token.head)
            if token.head.pos_ in ['NOUN', 'PROPN']:
                aspecto    = token.head.lemma_.lower()
                polaridade = lexico.get(adjetivo, 0)

                # Verificar negação nos filhos do substantivo
                for filho in token.head.children:
                    if filho.dep_ == 'advmod' and filho.lemma_.lower() in negs:
                        polaridade = -polaridade

                # Verificar negação direta no adjetivo
                for filho in token.children:
                    if filho.dep_ == 'advmod' and filho.lemma_.lower() in negs:
                        polaridade = -polaridade

                if polaridade != 0:
                    sentimento = 'POSITIVO' if polaridade > 0 else 'NEGATIVO'
                    aspectos.append((aspecto, adjetivo, sentimento))

    return aspectos

print('Função extrair_aspectos definida!')

### 3.3 — Testando o ABSA

In [ ]:
reviews_absa = [
    'A câmera é excelente mas a bateria é fraca.',
    'O design é bonito e o processador é rápido, mas o preço está caro.',
    'A entrega foi rápida porém o produto chegou quebrado.',
    'Atendimento péssimo mas o produto é muito bom.',
    'A tela é bonita e o som é excelente. A bateria é fraca.',
]

print('=== ABSA — Análise por Aspecto ===\n')

for review in reviews_absa:
    resultados = extrair_aspectos(review, lexico_sent, negadores)
    print(f'Review: "{review}"')
    if resultados:
        for aspecto, adjetivo, sentimento in resultados:
            emoji = '✅' if sentimento == 'POSITIVO' else '❌'
            print(f'  {emoji} {aspecto:<15} → "{adjetivo}" ({sentimento})')
    else:
        print('  (nenhum aspecto identificado com léxico atual)')
    print()

### 3.4 — Comparativo: léxico vs. supervisionado vs. ABSA

In [ ]:
# Comparar as três abordagens na mesma review
review_comp = 'A câmera é excelente mas a bateria é muito fraca.'

# Abordagem 1: Léxica
classe_lex, escore_lex = classificar_lexico(
    review_comp, lexico_positivo, lexico_negativo, amplificadores, negadores
)

# Abordagem 2: Supervisionada
proc       = preprocessar(review_comp)
vec        = vetorizador.transform([proc])
pred_sup   = lr.predict(vec)[0]
proba_sup  = lr.predict_proba(vec)[0]
classe_sup = 'POSITIVO' if pred_sup == 1 else 'NEGATIVO'
conf_sup   = max(proba_sup)

# Abordagem 3: ABSA
aspectos   = extrair_aspectos(review_comp, lexico_sent, negadores)

print(f'Review: "{review_comp}"')
print()
print('Abordagem 1 — Léxica:')
print(f'  → {classe_lex} (escore: {escore_lex:+.1f})')
print()
print('Abordagem 2 — Supervisionada (Regressão Logística):')
print(f'  → {classe_sup} (confiança: {conf_sup:.1%})')
print()
print('Abordagem 3 — ABSA:')
for aspecto, adjetivo, sentimento in aspectos:
    emoji = '✅' if sentimento == 'POSITIVO' else '❌'
    print(f'  {emoji} {aspecto:<12} → {adjetivo} ({sentimento})')
print()
print('Qual abordagem capturou melhor a complexidade da review?')

### 📝 Reflexão — Parte 3

**Pergunta 1:** Para a review `'A câmera é excelente mas a bateria é fraca'`,
qual das três abordagens capturou melhor o sentimento real? Por quê?

**Pergunta 2:** Imagine que você trabalha no setor de QA de uma empresa de smartphones.
Qual abordagem seria mais útil para o seu trabalho — documento, sentença ou aspecto? Justifique.

**Pergunta 3:** O extrator de aspectos não identificou nada na frase
`'Chegou antes do prazo!'`. Qual é o motivo técnico desse resultado?

*(Responda aqui)*

---
# 📌 PARTE 4 — Pipeline Completo com Novas Reviews

Vamos aplicar as três abordagens a reviews reais que o modelo **nunca viu**.

In [ ]:
novas_reviews = [
    'Produto chegou muito rápido e funciona perfeitamente!',
    'Péssima compra. Quebrou na primeira semana de uso.',
    'Não gostei. Qualidade muito abaixo do esperado.',
    'Bom produto pelo preço. Recomendo para quem busca custo-benefício.',
    'Entrega demorou mas o produto vale a pena.',
    'O design é bonito mas o material é muito frágil.',
]

print('=== Classificação de Novas Reviews ===')
print()
print(f'{"#":<3} {"Review":<45} {"Léxico":<11} {"Supervisionado":<18} {"Conf."}')
print('-' * 88)

for i in range(len(novas_reviews)):
    review = novas_reviews[i]

    # Léxico
    cl, _ = classificar_lexico(
        review, lexico_positivo, lexico_negativo, amplificadores, negadores
    )

    # Supervisionado
    proc  = preprocessar(review)
    vec   = vetorizador.transform([proc])
    pred  = lr.predict(vec)[0]
    proba = lr.predict_proba(vec)[0]
    cs    = 'POSITIVO' if pred == 1 else 'NEGATIVO'
    conf  = max(proba)

    print(f'{i+1:<3} {review[:44]:<45} {cl:<11} {cs:<18} {conf:.0%}')

print()
print('Onde as duas abordagens discordam? Qual você confia mais?')

---
# ✏️ Atividade Prática

Agora é a sua vez! Escolha um domínio diferente de reviews de produtos
(filmes, restaurantes, aplicativos, livros, jogos) e aplique o pipeline completo.

**O que você vai fazer:**
1. Criar um corpus com pelo menos 10 reviews rotuladas do seu tema
2. Aplicar o pré-processamento
3. Treinar um classificador com TF-IDF + Regressão Logística
4. Avaliar com as métricas
5. Identificar os aspectos mais recorrentes com ABSA
6. Responder às perguntas

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  PASSO 1 — Crie seu corpus aqui!        ║
# ╚══════════════════════════════════════════╝

meu_corpus = [
    # (texto, rotulo)  — 1 = positivo, 0 = negativo
    # Insira pelo menos 10 reviews do seu tema
    ('substitua por uma review positiva do seu tema', 1),
    ('substitua por outra review positiva', 1),
    ('mais uma review positiva', 1),
    ('mais uma review positiva diferente', 1),
    ('última review positiva', 1),
    ('substitua por uma review negativa do seu tema', 0),
    ('substitua por outra review negativa', 0),
    ('mais uma review negativa', 0),
    ('mais uma review negativa diferente', 0),
    ('última review negativa', 0),
]

meus_textos  = [r[0] for r in meu_corpus]
meus_rotulos = [r[1] for r in meu_corpus]

print(f'Corpus criado: {len(meu_corpus)} reviews')
print(f'  Positivas: {sum(meus_rotulos)}')
print(f'  Negativas: {len(meus_rotulos) - sum(meus_rotulos)}')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  PASSO 2 — Pré-processamento            ║
# ╚══════════════════════════════════════════╝

meus_textos_proc = []
for texto in meus_textos:
    meus_textos_proc.append(preprocessar(texto))

print('Após pré-processamento:')
for i in range(min(4, len(meus_textos_proc))):
    emoji = '✅' if meus_rotulos[i] == 1 else '❌'
    print(f'  {emoji} "{meus_textos_proc[i]}"')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  PASSO 3 — TF-IDF + Classificador       ║
# ╚══════════════════════════════════════════╝

if len(meu_corpus) >= 8:

    meu_X_train, meu_X_test, meu_y_train, meu_y_test = train_test_split(
        meus_textos_proc, meus_rotulos,
        test_size=0.3, random_state=42
    )

    meu_vet = TfidfVectorizer(ngram_range=(1, 2))
    meu_X_tr = meu_vet.fit_transform(meu_X_train)
    meu_X_te = meu_vet.transform(meu_X_test)

    meu_lr = LogisticRegression(random_state=42, max_iter=1000)
    meu_lr.fit(meu_X_tr, meu_y_train)
    meu_pred = meu_lr.predict(meu_X_te)

    print('=== Resultados no seu corpus ===')
    print(classification_report(
        meu_y_test, meu_pred,
        target_names=['Negativo', 'Positivo'],
        zero_division=0
    ))

else:
    print('Adicione pelo menos 8 reviews no corpus para treinar o classificador.')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  PASSO 4 — ABSA no seu corpus           ║
# ╚══════════════════════════════════════════╝

print('=== Aspectos identificados no seu corpus ===\n')

for texto in meus_textos[:5]:
    aspectos = extrair_aspectos(texto, lexico_sent, negadores)
    print(f'Review: "{texto[:60]}"')
    if aspectos:
        for aspecto, adjetivo, sentimento in aspectos:
            emoji = '✅' if sentimento == 'POSITIVO' else '❌'
            print(f'  {emoji} {aspecto:<15} → {adjetivo} ({sentimento})')
    else:
        print('  (nenhum aspecto identificado — tente expandir o léxico)')
    print()

---
### ❓ Perguntas da Atividade

**1.** Qual foi o F1-Score do classificador no seu corpus?
Esse valor é confiável dado o tamanho do corpus? Por quê?

**2.** Compare a abordagem léxica e a supervisionada em 2 reviews do seu corpus.
Houve discordância? Qual estava certa?

**3.** O ABSA identificou aspectos relevantes para o seu domínio?
Quais palavras do léxico precisariam ser adicionadas para melhorar os resultados?

**4.** Se o corpus fosse 10× maior, o classificador melhoraria?
Qual das métricas você esperaria que melhorasse mais?

**5.** No seu domínio escolhido, um falso negativo (positivo classificado como negativo)
ou um falso positivo (negativo classificado como positivo) seria mais prejudicial? Por quê?

*(Escreva suas respostas aqui)*

---
## ✅ Resumo da Aula 08

| Método | Ferramenta | O que faz |
|--------|------------|----------|
| **Léxico** | Dicionário manual | Soma escores de palavras pré-classificadas |
| **Supervisionado** | TF-IDF + Naive Bayes / Logística | Aprende padrões de exemplos rotulados |
| **ABSA** | spaCy (`pos_`, `head`, `dep_`) | Identifica sentimento por aspecto |

### Fórmulas essenciais

```
Precisão  = TP / (TP + FP)              → qualidade das previsões positivas
Recall    = TP / (TP + FN)              → cobertura dos positivos reais
F1-Score  = 2 × P × R / (P + R)        → equilíbrio entre P e R
Acurácia  = (TP + TN) / Total           → só confiável com classes balanceadas
```

### 🔗 Conexão com as próximas aulas

- **Aula 09 — NER:** identificar *sobre quem* é o sentimento — `'Apple lançou produto horrível'`
- **Aula 11 — Transformers:** BERTimbau supera TF-IDF + Logística com contexto bidirecional
- **Projeto final:** pipeline completo de análise de sentimento por aspecto em domínio real

---

### 📚 Referências

- **B1** Caseli & Nunes. *PLN: conceitos, técnicas e aplicações em português*. BPLN, 2023. Cap. 9.
- **SentiLex-PT:** INESC-ID Lisboa — léxico de sentimento para português
- **scikit-learn:** [scikit-learn.org](https://scikit-learn.org) — Naive Bayes, Logística, métricas

---
*IBM3130 · PLN · Aula 08 · 2º Semestre de 2026*